# Figure 1c: Dataset And Task Scope

- Figure / panel: `Fig1c`
- Inputs: processed dataset `h5ad`
- Outputs: `artifacts/paper_figures/main/Fig1_MethodOverview/fig1_dataset_scope.png`, `fig1_dataset_scope.csv`
- Run order: run before all result notebooks; this notebook only prepares the dataset/task scope panel for Figure 1
- `Fig1a`: task reframing schematic, generated by external figure AI
- `Fig1b`: TriShift framework schematic, generated by external figure AI
- Role: Main text


In [2]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / 'scripts').exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
if str(repo_root / 'src') not in sys.path:
    sys.path.insert(0, str(repo_root / 'src'))

from trishift._utils import load_adata, load_yaml


## Plan

- `Fig1a` and `Fig1b` are external schematics. This notebook no longer prepares data for those panels.
- `Fig1c` is a compact dataset summary table, aligned with the role played by Scouter Fig. 1d.
- The current default scope includes three datasets: `Adamson`, `Norman`, and `Dixit`.
- Only the first three columns are retained: dataset, number of cells, and number of perturbations.


In [3]:
DATASET_SPECS = [
    {'dataset_key': 'dixit', 'dataset_label': 'Dixit'},
    {'dataset_key': 'adamson', 'dataset_label': 'Adamson'},
    {'dataset_key': 'norman', 'dataset_label': 'Norman'},
]
PATHS_CFG = load_yaml(str(repo_root / 'configs' / 'paths.yaml'))
OUT_ROOT = repo_root / 'artifacts' / 'paper_figures' / 'main' / 'Fig1_MethodOverview'
OUT_ROOT.mkdir(parents=True, exist_ok=True)

for stale_name in ['fig1_norman_subgroup_counts.csv', 'fig1_norman_subgroup_counts.png']:
    stale_path = OUT_ROOT / stale_name
    if stale_path.exists():
        stale_path.unlink()


def _repo_path(path_like):
    path = Path(path_like)
    return path if path.is_absolute() else repo_root / path


def perturbation_summary(dataset_key: str) -> tuple[int, str]:
    adata = load_adata(str(_repo_path(PATHS_CFG['datasets'][dataset_key])))
    uniq = sorted(set(adata.obs['condition'].astype(str)))
    uniq_nonctrl = [c for c in uniq if c != 'ctrl']
    one_gene = sum(1 for c in uniq_nonctrl if len(c.split('+')) == 2 and 'ctrl' in c.split('+'))
    two_gene = sum(1 for c in uniq_nonctrl if len(c.split('+')) == 2 and 'ctrl' not in c.split('+'))
    other = len(uniq_nonctrl) - one_gene - two_gene
    lines = []
    if one_gene:
        lines.append(f'{one_gene:,} one-gene')
    if two_gene:
        lines.append(f'{two_gene:,} two-gene')
    if other:
        lines.append(f'{other:,} other')
    return int(adata.n_obs), '\n'.join(lines) if lines else '0'


rows = []
for spec in DATASET_SPECS:
    n_cells, perturb_text = perturbation_summary(spec['dataset_key'])
    rows.append({
        'Dataset': spec['dataset_label'],
        'No. of cells': f'{n_cells:,}',
        'No. of perturbations': perturb_text,
    })

scope_df = pd.DataFrame(rows)
scope_df.to_csv(OUT_ROOT / 'fig1_dataset_scope.csv', index=False, encoding='utf-8-sig')
display(scope_df)


,Dataset,No. of cells,No. of perturbations
0,Dixit,"44,735",19 one-gene
1,Adamson,"68,603",86 one-gene
2,Norman,"91,205",105 one-gene\n131 two-gene


In [4]:
fig, ax = plt.subplots(figsize=(5.8, 3.3), dpi=240)
ax.axis('off')
table = ax.table(
    cellText=scope_df.values,
    colLabels=scope_df.columns,
    loc='center',
    cellLoc='center',
    colLoc='center',
)
table.auto_set_font_size(False)
table.set_fontsize(9.5)
table.scale(1.15, 2.0)

for (row, col), cell in table.get_celld().items():
    cell.set_edgecolor('#4A607A')
    cell.set_linewidth(0.8)
    if row == 0:
        cell.set_facecolor('#F1F4F8')
        cell.set_text_props(weight='normal')
    else:
        cell.set_facecolor('white')

plt.tight_layout()
plt.savefig(OUT_ROOT / 'fig1_dataset_scope.png', bbox_inches='tight', pad_inches=0.04)
plt.close(fig)
print(OUT_ROOT)


E:\CODE\trishift\artifacts\paper_figures\main\Fig1_MethodOverview
